# ECE 214B Project 2: Final-Final Colab Run

This notebook is a runner only. All scientific and modeling logic lives in `project/scripts/` and `project/src/`.

Workflow: upload `colab_run_final_final/` to Google Drive, rename it to `214B_colab_run_final_final`, edit `DATA_ZIP`, set `OPENAI_API_KEY` if running LLM sections, set GPU runtime, and run the workflow. The dataset zip is extracted to local Colab storage before experiments run.


## Setup

Edit the path variables below. `RUN_LLM=True` runs the prior clinical marker API panel. `RUN_LLM_SCENE_GRAPH=True` and `RUN_LLM_EVENT_TRIPLES=True` enable resumable Cookie Theft LLM sections, but the actual API batches are in manual cells so interrupted Colab sessions can resume safely from cache.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

DROP_DIR = "/content/drive/MyDrive/214B_colab_run_final_final"
DATA_ZIP = "/content/drive/MyDrive/path/to/S26_ECE_214B_Mini_Project_2.zip"
EXTRACT_DIR = "/content/214B_data"

RUN_ASR = True
RUN_LLM = True
RUN_LLM_SCENE_GRAPH = True
RUN_LLM_EVENT_TRIPLES = False

LLM_MODEL = "gpt-4o-mini"
LLM_SCENE_GRAPH_BATCH_SIZE = 50
LLM_EVENT_TRIPLE_BATCH_SIZE = 50

PROJECT_DIR = f"{DROP_DIR}/project"
OUTPUT_ZIP = f"{DROP_DIR}/outputs/outputs_run_final_final.zip"
SUMMARY_ZIP = f"{DROP_DIR}/outputs/final_summary_files.zip"

print("DROP_DIR:", DROP_DIR)
print("DATA_ZIP:", DATA_ZIP)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("RUN_ASR:", RUN_ASR)
print("RUN_LLM:", RUN_LLM)
print("RUN_LLM_SCENE_GRAPH:", RUN_LLM_SCENE_GRAPH)
print("RUN_LLM_EVENT_TRIPLES:", RUN_LLM_EVENT_TRIPLES)
print("LLM_SCENE_GRAPH_BATCH_SIZE:", LLM_SCENE_GRAPH_BATCH_SIZE)
print("LLM_EVENT_TRIPLE_BATCH_SIZE:", LLM_EVENT_TRIPLE_BATCH_SIZE)


In [ ]:
# Optional LLM setup.
# Prefer Colab secrets or a private runtime environment variable named OPENAI_API_KEY.
# Do not paste API keys into notebooks that may be shared or committed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

drop_path = Path(DROP_DIR)
project_path = Path(PROJECT_DIR)
data_zip_path = Path(DATA_ZIP)

if not drop_path.exists():
    raise FileNotFoundError(f"DROP_DIR not found: {drop_path}")
if not project_path.exists():
    raise FileNotFoundError(f"PROJECT_DIR not found: {project_path}")
for required in [project_path / "scripts", project_path / "src", project_path / "requirements.txt"]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required project item: {required}")
if not data_zip_path.exists():
    raise FileNotFoundError(f"DATA_ZIP not found. Edit DATA_ZIP: {data_zip_path}")
if (RUN_LLM or RUN_LLM_SCENE_GRAPH or RUN_LLM_EVENT_TRIPLES) and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LLM run requested but OPENAI_API_KEY is not set. Use the optional key cell or Colab secrets.")

print("Drive/package validation complete.")


## Dataset Extraction

The dataset zip is extracted to local Colab storage. Existing project outputs are not deleted.


In [ ]:
extract_path = Path(EXTRACT_DIR)
if extract_path.exists():
    shutil.rmtree(extract_path)
extract_path.mkdir(parents=True, exist_ok=True)

print("Extracting", DATA_ZIP, "to", EXTRACT_DIR)
with zipfile.ZipFile(DATA_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

def find_dataset_dir(root: Path) -> Path:
    candidates = []
    for path in [root, *root.rglob("*")]:
        if path.is_dir() and all((path / name).exists() for name in ["splits", "sessions", "wavs"]):
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError("Could not find extracted dataset folder containing splits/, sessions/, wavs/.")
    return sorted(candidates, key=lambda p: len(p.parts))[0]

DATA_DIR = str(find_dataset_dir(extract_path))
print("DATA_DIR:", DATA_DIR)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print("Python:", sys.version)
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: no GPU detected. CPU experiments can run; transformer/audio experiments may be slow.")
except Exception as exc:
    print("torch import failed:", repr(exc))


## Core Baselines


In [ ]:
core_commands = [
    [sys.executable, "scripts/audit_dataset.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_descriptor_baselines.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_acoustic_descriptor_baseline.py", "--data_dir", DATA_DIR],
]
for command in core_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Transformer Baselines

Full HuBERT layer sweep is intentionally not run here.


In [ ]:
transformer_commands = [
    [sys.executable, "scripts/run_roberta_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_hubert_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_wavlm_baseline.py", "--data_dir", DATA_DIR],
]
for command in transformer_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Fusion And Clinical Thresholding


In [ ]:
fusion_commands = [
    [sys.executable, "scripts/run_fusion.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_clinical_thresholding.py", "--data_dir", DATA_DIR],
]
for command in fusion_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Phase 2 Interpretability


In [ ]:
phase2_commands = [
    [sys.executable, "scripts/run_linguistic_marker_panel.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/analyze_marker_errors.py", "--data_dir", DATA_DIR],
]
for command in phase2_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Phase 3 Hybrid/Clinical Analyses


In [ ]:
phase3_commands = [
    [sys.executable, "scripts/run_uncertainty_gated_marker_fusion.py"],
    [sys.executable, "scripts/run_marker_error_corrector.py"],
    [sys.executable, "scripts/run_stacked_tfidf_marker_meta_model.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_marker_stratified_thresholding.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_age_aware_thresholding.py", "--data_dir", DATA_DIR],
]
for command in phase3_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## ASR Experiments


In [ ]:
if RUN_ASR:
    asr_commands = [
        [sys.executable, "scripts/run_asr_error_features.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/asr_error_features", "--whisper_model", "base"],
        [sys.executable, "scripts/run_asr_transcript_text_fusion.py", "--data_dir", DATA_DIR],
    ]
    for command in asr_commands:
        print("+", " ".join(command))
        subprocess.run(command, check=True)
else:
    print("Skipping ASR experiments because RUN_ASR=False")


## LLM Clinical Marker Experiments


In [ ]:
if RUN_LLM:
    llm_commands = [
        [sys.executable, "scripts/run_llm_clinical_marker_panel.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/llm_clinical_marker_panel", "--model", LLM_MODEL],
        [sys.executable, "scripts/run_llm_stratified_thresholding.py", "--data_dir", DATA_DIR],
        [sys.executable, "scripts/run_llm_fusion_optimization.py", "--data_dir", DATA_DIR],
    ]
    for command in llm_commands:
        print("+", " ".join(command))
        subprocess.run(command, check=True)
else:
    print("Skipping LLM clinical marker experiments because RUN_LLM=False")


## LLM Cookie Theft Scene-Graph Scorer — Resumable Batches

This section is manual/resumable by design. The cache lives at `outputs/llm_cookie_theft_scene_graph/raw_responses_cache.jsonl` inside the Drive-backed project folder. Do not delete `outputs/` between batches.

Run the progress cell first. Then repeatedly run the "next missing batch" cell until progress reports `Missing sessions: 0`. Finally run the cached modeling pass and print result files.


In [ ]:
if RUN_LLM_SCENE_GRAPH:
    command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_scene_graph",
        "--model", LLM_MODEL,
        "--list_progress",
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Skipping scene-graph progress check because RUN_LLM_SCENE_GRAPH=False")


In [ ]:
# Run this cell repeatedly until the progress cell reports Missing sessions: 0.
if RUN_LLM_SCENE_GRAPH:
    command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_scene_graph",
        "--model", LLM_MODEL,
        "--only_missing",
        "--batch_size", str(LLM_SCENE_GRAPH_BATCH_SIZE),
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Skipping scene-graph batch because RUN_LLM_SCENE_GRAPH=False")


In [ ]:
# Run after all scene-graph sessions are cached. If any are still missing, this updates partial CSVs and skips modeling.
if RUN_LLM_SCENE_GRAPH:
    command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_scene_graph",
        "--model", LLM_MODEL,
        "--only_missing",
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)

    for rel in [
        "outputs/llm_cookie_theft_scene_graph/results.md",
        "outputs/llm_cookie_theft_scene_graph/fusion_results.csv",
        "outputs/llm_cookie_theft_scene_graph/top_scene_graph_coefficients.csv",
    ]:
        path = Path(rel)
        print("\n====", rel, "====")
        if path.exists():
            print(path.read_text()[:8000])
        else:
            print("Missing:", rel)
else:
    print("Skipping scene-graph final modeling because RUN_LLM_SCENE_GRAPH=False")


## LLM Cookie Theft Event-Triple Scorer — Resumable Batches

This optional section extracts task-specific event triples such as who did what, to what object, and where/how. It is manual/resumable by design. The cache lives at `outputs/llm_cookie_theft_event_triples/raw_responses_cache.jsonl` inside the Drive-backed project folder.

Run the progress cell first. Then repeatedly run the "next missing batch" cell until progress reports `Missing sessions: 0`. Finally run the cached modeling pass and print result files.


In [ ]:
if RUN_LLM_EVENT_TRIPLES:
    command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_event_triples.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_event_triples",
        "--model", LLM_MODEL,
        "--list_progress",
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Skipping event-triple progress check because RUN_LLM_EVENT_TRIPLES=False")


In [ ]:
# Run this cell repeatedly until the progress cell reports Missing sessions: 0.
if RUN_LLM_EVENT_TRIPLES:
    command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_event_triples.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_event_triples",
        "--model", LLM_MODEL,
        "--only_missing",
        "--batch_size", str(LLM_EVENT_TRIPLE_BATCH_SIZE),
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Skipping event-triple batch because RUN_LLM_EVENT_TRIPLES=False")


In [ ]:
# Run after all event-triple sessions are cached. If any are still missing, this updates partial CSVs and skips modeling.
if RUN_LLM_EVENT_TRIPLES:
    command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_event_triples.py",
        "--data_dir", DATA_DIR,
        "--out_dir", "outputs/llm_cookie_theft_event_triples",
        "--model", LLM_MODEL,
        "--only_missing",
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)

    for rel in [
        "outputs/llm_cookie_theft_event_triples/results.md",
        "outputs/llm_cookie_theft_event_triples/fusion_results.csv",
        "outputs/llm_cookie_theft_event_triples/top_event_triple_coefficients.csv",
    ]:
        path = Path(rel)
        print("\n====", rel, "====")
        if path.exists():
            print(path.read_text()[:8000])
        else:
            print("Missing:", rel)
else:
    print("Skipping event-triple final modeling because RUN_LLM_EVENT_TRIPLES=False")


## Final Result Collection And Packaging


In [ ]:
subprocess.run([sys.executable, "scripts/collect_required_results.py"], check=True)

output_zip = Path(OUTPUT_ZIP)
summary_zip = Path(SUMMARY_ZIP)
output_zip.parent.mkdir(parents=True, exist_ok=True)
for path in [output_zip, summary_zip]:
    if path.exists():
        path.unlink()

shutil.make_archive(str(output_zip.with_suffix("")), "zip", root_dir=PROJECT_DIR, base_dir="outputs")

reports_dir = Path(PROJECT_DIR) / "reports"
if reports_dir.exists():
    with zipfile.ZipFile(output_zip, "a", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in reports_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(PROJECT_DIR))

summary_files = [
    "outputs/required_results_summary/results_table.md",
    "outputs/required_results_summary/summary.md",
    "outputs/fusion/results.md",
    "outputs/hubert_layer_sweep/results.md",
    "outputs/clinical_thresholding/results.md",
    "outputs/asr_error_features/results.md",
    "outputs/asr_error_profile_fusion/results.md",
    "outputs/asr_transcript_text_fusion/results.md",
    "outputs/llm_clinical_marker_panel/results.md",
    "outputs/llm_stratified_thresholding/results.md",
    "outputs/llm_fusion_optimization/results.md",
    "outputs/llm_cookie_theft_scene_graph/results.md",
    "outputs/llm_cookie_theft_scene_graph/fusion_results.csv",
    "outputs/llm_cookie_theft_scene_graph/top_scene_graph_coefficients.csv",
    "outputs/llm_cookie_theft_event_triples/results.md",
    "outputs/llm_hard_case_adjudicator/results.md",
    "outputs/scene_graph_fusion_optimization/results.md",
    "outputs/scene_graph_winner_analysis/results.md",
    "outputs/final_push_no_api/results.md",
    "outputs/literature_inspired_final_push/results.md",
    "outputs/llm_multiview_scene_profile/results.md",
    "outputs/llm_multiview_evidence_profile/results.md",
    "outputs/llm_scene_view_aggregation/results.md",
    "outputs/statistical_significance_tests/results.md",
    "outputs/statistical_significance_official_baseline/results.md",
    "outputs/scene_graph_feature_subset_search/results.md",
    "outputs/final_calibrated_scene_fusion/results.md",
    "outputs/final_decision_structure_experiments/results.md",
    "outputs/disagreement_aware_scene_fusion/results.md",
    "outputs/cascade_triage_experiment/results.md",
    "outputs/transcript_view_aggregation/results.md",
    "outputs/pause_disfluency_fusion/results.md",
    "outputs/two_pass_uncertainty_resolver/results.md",
    "outputs/tfidf_plus_scene_ablation/results.md",
    "outputs/semantic_specificity_fusion/results.md",
    "outputs/subject_feature_aggregation/results.md",
    "outputs/metadata_clinical_validation/results.md",
    "outputs/linguistic_marker_panel/results.md",
    "outputs/linguistic_marker_panel/error_analysis/marker_error_analysis_summary.md",
    "outputs/uncertainty_gated_marker_fusion/results.md",
    "outputs/marker_error_corrector/results.md",
    "outputs/stacked_tfidf_marker_meta_model/results.md",
    "outputs/marker_stratified_thresholding/results.md",
    "outputs/age_aware_thresholding/results.md",
]
with zipfile.ZipFile(summary_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in summary_files:
        path = Path(PROJECT_DIR) / rel
        if path.exists():
            zf.write(path, rel)

print("Full output zip:", output_zip)
print("Full output zip size MB:", round(output_zip.stat().st_size / (1024 * 1024), 2))
print("Summary zip:", summary_zip)
print("Summary zip size KB:", round(summary_zip.stat().st_size / 1024, 1))


## Optional Slow HuBERT Layer Sweep

Full HuBERT layer sweep is intentionally not run by default. Uncomment and run manually only if needed.


In [ ]:
# Optional: full HuBERT layer sweep, slow. Not run by default.
# subprocess.run([sys.executable, "scripts/run_hubert_layer_sweep.py", "--data_dir", DATA_DIR], check=True)
